# Computer lab 3: duality

**Mathematical Optimization, VU Amsterdam, 2026–27.**

Derive each dual before implementing it. Keep track of the objective sense,
constraint orientation and variable domains. Check primal and dual feasibility
as well as their objective values, and explain what the multipliers mean.

**Solutions.** Numerical results below are produced by the code; exact certificates are labeled separately.

## Setup

Run the next two cells before starting. They prepare HiGHS for Exercises 1 and
3–5 and Ipopt for Exercise 2.
If asked to restart the session, restart and run Setup again. See the
[course setup instructions](https://github.com/alessandrozocca/MO2026#computer-lab-setup)
for local use and troubleshooting.

In [1]:
#@title Prepare course helpers and solvers
import hashlib
import os
from pathlib import Path
import sys
import time
import types
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

SETUP_COMMIT = '425c3f0af23e082c276df85fcde556538cc2abef'
SETUP_SHA256 = "f742478c6eaf5c5db82f4ce69b47a2adcf6f920765d88c3e7eb1b1ab72e66d28"
local_repo = os.environ.get("MO2026_REPO")
if local_repo:
    setup_location = str(Path(local_repo).expanduser() / "lab_setup.py")
    setup_source = Path(setup_location).read_bytes()
else:
    setup_location = f"https://raw.githubusercontent.com/alessandrozocca/MO2026/{SETUP_COMMIT}/lab_setup.py"
    for attempt in range(3):
        try:
            with urlopen(setup_location, timeout=30) as response:
                setup_source = response.read()
            break
        except (URLError, TimeoutError, ConnectionError) as error:
            if (isinstance(error, HTTPError) and error.code not in (408, 429, 500, 502, 503, 504)) or attempt == 2:
                raise
            time.sleep(0.5 * (attempt + 1))
    if hashlib.sha256(setup_source).hexdigest() != SETUP_SHA256:
        raise RuntimeError("Course setup checksum mismatch; the downloaded code was not executed.")

lab_setup = types.ModuleType("mo2026_lab_setup")
lab_setup.__file__ = setup_location
sys.modules[lab_setup.__name__] = lab_setup
exec(compile(setup_source, setup_location, "exec"), lab_setup.__dict__)
if local_repo and lab_setup.API_VERSION != 1:
    raise RuntimeError("This notebook requires course setup API 1.")
lab = lab_setup.setup(week=3, check=False, verbose=True)
solve_and_load = lab_setup.solve_and_load
format_number = lambda value, digits=4: lab_setup.format_number(value, digits)
lp_solver, nlp_solver = lab.lp_solver, lab.nlp_solver
dual_display = lab.helpers['tutorial3']
add_dual_variables = dual_display.add_dual_variables
display_dual_variables = dual_display.display_dual_variables

Week 3 ready: Pyomo 6.10.1; HiGHS 1.15.1; Ipopt 3.14.20 (Darwin arm64), ASL(20241202).


In [2]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from IPython.display import display

Useful references are the MO-book's [dual of the BIM production problem](https://mobook.github.io/MO-book/notebooks/02/06-bim-dual.html)
and the [Pyomo suffix documentation](https://pyomo.readthedocs.io/en/6.10.1/explanation/modeling/math_programming/suffixes.html).
The BIM example is a maximization primal; the two primals below are minimization
problems, so check their signs explicitly.

## Imported row duals and Lagrangian multipliers

For a **minimization primal**, the solvers used here report a raw RHS dual $d_i$
for each constraint row:

| Row orientation | Sign of raw RHS dual |
|:---|:---|
| $a_i^\top x\ge b_i$ | $d_i\ge0$ |
| $a_i^\top x\le b_i$ | $d_i\le0$ |
| $a_i^\top x=b_i$ | $d_i$ unrestricted |

Where the optimal-value function is differentiable, $d_i$ is its sensitivity to
increasing the RHS $b_i$ of that row. Here "RHS" means the row's bound $b_i$,
wherever Pyomo prints it. Negating a row changes the reported dual sign.
Exercise 1 uses LP dual variables $y_i$ in this RHS convention.

In Exercise 2 we write every inequality as $g_i(x)\le0$ and use
$L(x,\lambda)=f(x)+\sum_i\lambda_i g_i(x)$ with $\lambda_i\ge0$.
For these upper-bound rows, **$\lambda_i=-d_i$**. Keep the two sign conventions
separate when comparing imported values with a hand-derived dual.

Attach `model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)` **before solving**.
After a successful solve, read `model.dual[model.row[i]]` for an indexed row.
The imported course function `add_dual_variables(model)` also attaches this suffix.

As in weeks 1–2, `solve_and_load` checks termination before loading a solution.
Pass the solver explicitly and then evaluate with `pyo.value(...)`. Ipopt checks
local optimality conditions; convexity and the primal–dual certificates below
justify global conclusions here.

## Exercise 1: a linear primal and its dual

Consider the LP

$$
\begin{aligned}
\min\quad &x_1-2x_2+4x_3\\
\text{s.t.}\quad &-x_1+3x_2=3 &&(1),\\
&2x_1-x_2+3x_3\ge-5 &&(2),\\
&x_3\le7 &&(3),\\
&x_1+x_2-x_3\le11 &&(4),\\
&x_1,x_2,x_3\in\mathbb{R}.
\end{aligned}
$$

**(a)** Implement and solve the primal with HiGHS. Keep all three variables free
and attach a dual suffix before solving.

**(b)** Derive the dual by hand. Explain the sign of each dual variable and why
the dual constraints are equalities. Implement and solve the dual with HiGHS.

**(c)** Read the imported row duals directly from the primal suffix and compare
them with your explicit dual solution. You can also use
`display_dual_variables(model)` to display the rows and their raw duals.
Does strong duality hold, and could you have predicted it? Interpret the signs
of the nonzero RHS duals in this example.

In [3]:
linear_rhs = {1: 3, 2: -5, 3: 7, 4: 11}

### Solution to 1(a)

Use real-valued variables, allowing $x_3$ to be negative, and keep the original
row orientations. Indexing the rows makes their correspondence with the four
dual variables explicit.

In [4]:
def build_linear_primal(rhs):
    model = pyo.ConcreteModel(name="Exercise 1 primal")
    model.x = pyo.Var([1, 2, 3], domain=pyo.Reals)
    model.objective = pyo.Objective(expr=model.x[1] - 2 * model.x[2] + 4 * model.x[3])
    model.row = pyo.Constraint([1, 2, 3, 4])
    model.row[1] = -model.x[1] + 3 * model.x[2] == rhs[1]
    model.row[2] = 2 * model.x[1] - model.x[2] + 3 * model.x[3] >= rhs[2]
    model.row[3] = model.x[3] <= rhs[3]
    model.row[4] = model.x[1] + model.x[2] - model.x[3] <= rhs[4]
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return model

linear_primal = build_linear_primal(linear_rhs)
result = solve_and_load(linear_primal, lp_solver)
linear_point = np.array([pyo.value(linear_primal.x[j]) for j in [1, 2, 3]])
linear_primal_value = pyo.value(linear_primal.objective)
print(f"Termination: {result.solver.termination_condition}")
display(pd.Series(linear_point, index=["x1", "x2", "x3"], name="Primal value")
        .to_frame().style.format(format_number))
print(f"Primal objective: {format_number(linear_primal_value)}")

Termination: optimal


,Primal value
x1,4.5882
x2,2.5294
x3,-3.8824


Primal objective: -16.0000


### Solution to 1(b)

The dual in the RHS convention is

$$
\begin{aligned}
\max\quad &3y_1-5y_2+7y_3+11y_4\\
\text{s.t.}\quad &-y_1+2y_2+y_4=1,\\
&3y_1-y_2+y_4=-2,\\
&3y_2+y_3-y_4=4,\\
&y_1\in\mathbb{R},\quad y_2\ge0,\quad y_3,y_4\le0.
\end{aligned}
$$

Here $c=(1,-2,4)^\top$, $b=(3,-5,7,11)^\top$, and $A\in\mathbb{R}^{4\times3}$
has rows $a_i^\top$ given by the coefficients of the four primal constraints.
Each sign makes $y_i(a_i^\top x-b_i)\ge0$ for every primal feasible $x$:
row 1 is an equality, so $y_1$ is free; row 2 has $a_2^\top x-b_2\ge0$, so
$y_2\ge0$; rows 3–4 have $a_i^\top x-b_i\le0$, so $y_3,y_4\le0$.

The LP Lagrangian is

$$
L(x,y)=c^\top x-y^\top(Ax-b)=b^\top y+(c-A^\top y)^\top x.
$$

Its infimum over unrestricted $x\in\mathbb{R}^3$ is $-\infty$ unless
$A^\top y=c$, which explains the dual equalities. For any primal–dual feasible pair,

$$
c^\top x=y^\top Ax=\sum_i y_i a_i^\top x
\ge\sum_i y_i b_i=b^\top y.
$$

Thus the dual supplies a lower bound on the primal minimum (weak duality).

In [5]:
def build_linear_dual(rhs):
    model = pyo.ConcreteModel(name="Exercise 1 dual")
    model.y1 = pyo.Var(domain=pyo.Reals)
    model.y2 = pyo.Var(domain=pyo.NonNegativeReals)
    model.y3 = pyo.Var(domain=pyo.NonPositiveReals)
    model.y4 = pyo.Var(domain=pyo.NonPositiveReals)
    model.objective = pyo.Objective(
        expr=rhs[1] * model.y1 + rhs[2] * model.y2 + rhs[3] * model.y3 + rhs[4] * model.y4,
        sense=pyo.maximize,
    )
    model.stationarity = pyo.Constraint([1, 2, 3])
    model.stationarity[1] = -model.y1 + 2 * model.y2 + model.y4 == 1
    model.stationarity[2] = 3 * model.y1 - model.y2 + model.y4 == -2
    model.stationarity[3] = 3 * model.y2 + model.y3 - model.y4 == 4
    return model

linear_dual = build_linear_dual(linear_rhs)
result = solve_and_load(linear_dual, lp_solver)
linear_explicit_y = np.array([pyo.value(linear_dual.y1), pyo.value(linear_dual.y2),
                              pyo.value(linear_dual.y3), pyo.value(linear_dual.y4)])
linear_dual_value = pyo.value(linear_dual.objective)
print(f"Termination: {result.solver.termination_condition}")
display(pd.Series(linear_explicit_y, index=range(1, 5), name="Explicit LP dual y")
        .rename_axis("Row").to_frame().style.format(format_number))
print(f"Dual objective: {format_number(linear_dual_value)}")

Termination: optimal


,Explicit LP dual y
Row,
1,0.0000
2,1.0000
3,0.0000
4,-1.0000


Dual objective: -16.0000


### Solution to 1(c)

The imported RHS duals use exactly the convention of $y$ in this explicit LP dual.
Read each indexed constraint **data object**, not the entire indexed component.

In [6]:
linear_imported_d = np.array([linear_primal.dual[linear_primal.row[i]] for i in range(1, 5)])
display(pd.DataFrame(
    {"Explicit dual y": linear_explicit_y, "Imported RHS dual d": linear_imported_d},
    index=pd.Index(range(1, 5), name="Row"),
).style.format(format_number))
print(f"Primal minus dual objective: {linear_primal_value - linear_dual_value:.2e}")
display_dual_variables(linear_primal)

,Explicit dual y,Imported RHS dual d
Row,,
1,0.0000,0.0000
2,1.0000,1.0000
3,0.0000,0.0000
4,-1.0000,-1.0000


Primal minus dual objective: 0.00e+00


**Constraint duals (raw RHS convention):**

`row[1]`: `- x[1] + 3*x[2]  ==  3`; raw dual **0.000**.

`row[2]`: `-5  <=  2*x[1] - x[2] + 3*x[3]`; raw dual **1.000**.

`row[3]`: `x[3]  <=  7`; raw dual **0.000**.

`row[4]`: `x[1] + x[2] - x[3]  <=  11`; raw dual **-1.000**.

An exact certificate is $x^*=(78,43,-66)/17$ and $y^*=(0,1,0,-1)$.
The primal is feasible, and every feasible point satisfies

$$
x_1-2x_2+4x_3
=(2x_1-x_2+3x_3)-(x_1+x_2-x_3)\ge-5-11=-16.
$$

The point attains the bound, proving optimality without relying on the numerical
gap. The explicit dual is feasible and attains $-16$. LP strong duality follows
from feasibility and a finite optimal value, both established here by the point
and the bound.

Locally, increasing the RHS of row 2 tightens a lower bound and increases the
optimal value at rate $y_2=1$. Increasing the RHS of row 4 relaxes an upper bound
and decreases the optimal value at rate $y_4=-1$.
The equality row has $y_1=0$: a small change to $b_1$ leaves the optimal value
unchanged here. An equality constraint need not have a nonzero value sensitivity.

## Exercise 2: a convex quadratic primal and its Lagrangian dual

Consider

$$
\begin{aligned}
\min\quad &f(x)=x_1^2+x_2^2+x_3^2\\
\text{s.t.}\quad &g_1(x)=2x_1+x_2-5\le0,\\
&g_2(x)=x_1+x_3-2\le0,\\
&g_3(x)=1-x_1\le0,\\
&g_4(x)=2-x_2\le0,\\
&g_5(x)=-x_3\le0,\qquad x\in\mathbb{R}^3.
\end{aligned}
$$

**(a)** Explain convexity, implement the primal and solve it with Ipopt. Keep the
five inequalities in the stated $g_i(x)\le0$ orientation. Import the raw row duals
and convert them to the nonnegative multipliers in $L=f+\sum_i\lambda_i g_i$.

**(b)** Derive the Lagrangian dual by minimizing $L$ over the unrestricted vector
$x$. Implement and solve the resulting dual with Ipopt. Completing the square
gives a compact expression for the dual function.

**(c)** Check strong duality and explain whether it was predictable. Verify
complementary slackness and identify the binding constraints. Can a binding
constraint have a zero multiplier in this example?

In [7]:
# Matrix form of the five stated inequalities: g(x) = A @ x - b <= 0.
quadratic_A = np.array([
    [2, 1, 0],
    [1, 0, 1],
    [-1, 0, 0],
    [0, -1, 0],
    [0, 0, -1],
], dtype=float)
quadratic_b = np.array([5, 2, -1, -2, 0], dtype=float)

### Solution to 2(a)

The objective has Hessian $2I\succ0$ and every constraint is affine, so this is
a strictly convex primal problem. Keep the variables in $x$ unrestricted and
retain all five constraints as explicit rows, since the exercise asks for each
row's multiplier. Moving $g_3,g_4,g_5$ into variable bounds would
change where their multipliers are reported.

The code starts from the strictly feasible point $(1.25,2.25,0.5)$.
Numerical values below are approximate; part (c) supplies an exact certificate.

In [8]:
def build_quadratic_primal(A, b, start=None):
    n_rows, n_variables = A.shape
    if start is None:
        start = np.zeros(n_variables)
    model = pyo.ConcreteModel(name="Exercise 2 primal")
    model.variables = pyo.RangeSet(1, n_variables)
    model.rows = pyo.RangeSet(1, n_rows)
    model.x = pyo.Var(model.variables, domain=pyo.Reals,
                      initialize={j: start[j - 1] for j in range(1, n_variables + 1)})
    model.objective = pyo.Objective(expr=sum(model.x[j]**2 for j in model.variables))

    @model.Constraint(model.rows)
    def g(m, i):
        return sum(A[i - 1, j - 1] * m.x[j] for j in m.variables) - b[i - 1] <= 0

    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return model

quadratic_primal = build_quadratic_primal(quadratic_A, quadratic_b, start=np.array([1.25, 2.25, 0.5]))
result = solve_and_load(quadratic_primal, nlp_solver)
quadratic_point = np.array([pyo.value(quadratic_primal.x[j]) for j in quadratic_primal.variables])
quadratic_primal_value = pyo.value(quadratic_primal.objective)
quadratic_g = quadratic_A @ quadratic_point - quadratic_b
quadratic_imported_d = np.array([quadratic_primal.dual[quadratic_primal.g[i]] for i in quadratic_primal.rows])
quadratic_from_suffix = -quadratic_imported_d
print(f"Termination: {result.solver.termination_condition}")
display(pd.Series(quadratic_point, index=["x1", "x2", "x3"], name="Primal value")
        .to_frame().style.format(lambda value: format_number(value, 6)))
print(f"Primal objective: {quadratic_primal_value:.9f}")
display(pd.DataFrame(
    {"g_i(x)": quadratic_g, "Imported RHS dual d": quadratic_imported_d,
     "Lagrangian multiplier -d": quadratic_from_suffix},
    index=pd.Index([f"g_{i}" for i in quadratic_primal.rows], name="Row"),
).style.format(lambda value: format_number(value, 6)))

Termination: optimal


,Primal value
x1,1.000000
x2,2.000000
x3,0.000013


Primal objective: 5.000000000


,g_i(x),Imported RHS dual d,Lagrangian multiplier -d
Row,,,
g_1,-1.000000,0.000000,0.000000
g_2,-0.999987,0.000000,0.000000
g_3,0.000000,-2.000000,2.000000
g_4,0.000000,-4.000000,4.000000
g_5,-0.000013,-0.000027,0.000027


The nonzero raw upper-row duals are negative; the corresponding Lagrangian
multipliers are positive. If the required minimum in $g_3=a-x_1\le0$ changes,
then $dv/da=\lambda_3=-d_3$ locally.
The raw dual $d_3$ measures sensitivity to the RHS allowance of that coded row.

Ipopt can return small positive $x_3$ and a small positive approximation to
$\lambda_5$. The exact certificate in part (c) shows that both limiting values
are zero. Keep these approximations in calculations rather than replacing them
by rounded displayed values.

### Solution to 2(b)

With $g(x)=Ax-b$ and $\lambda\ge0$, the Lagrangian is

$$
L(x,\lambda)=x^\top x+\lambda^\top(Ax-b)
=\left\|x+\tfrac12A^\top\lambda\right\|_2^2
-\tfrac14\|A^\top\lambda\|_2^2-b^\top\lambda.
$$

The infimum over $x\in\mathbb{R}^3$ is attained at
$x(\lambda)=-\tfrac12A^\top\lambda$. Thus the dual is

$$
\max_{\lambda\ge0}\ell(\lambda),\qquad
\ell(\lambda)=-b^\top\lambda-\tfrac14\|A^\top\lambda\|_2^2.
$$

Its Hessian is $-\tfrac12AA^\top\preceq0$, so the dual function is concave.
For the given data this becomes

$$
\begin{aligned}
\ell(\lambda)={}&-5\lambda_1-2\lambda_2+\lambda_3+2\lambda_4\\
&-\tfrac14\left[(2\lambda_1+\lambda_2-\lambda_3)^2
+(\lambda_1-\lambda_4)^2+(\lambda_2-\lambda_5)^2\right].
\end{aligned}
$$

The code uses this squared-norm form as explicit indexed sums.

In [9]:
def build_quadratic_dual(A, b):
    n_rows, n_variables = A.shape
    model = pyo.ConcreteModel(name="Exercise 2 Lagrangian dual")
    model.rows = pyo.RangeSet(1, n_rows)
    model.variables = pyo.RangeSet(1, n_variables)
    model.lam = pyo.Var(model.rows, domain=pyo.NonNegativeReals, initialize=0.1)

    @model.Expression(model.variables)
    def adjoint(m, j):
        return sum(A[i - 1, j - 1] * m.lam[i] for i in m.rows)

    model.objective = pyo.Objective(
        expr=-sum(b[i - 1] * model.lam[i] for i in model.rows)
             - 0.25 * sum(model.adjoint[j]**2 for j in model.variables),
        sense=pyo.maximize,
    )
    return model

quadratic_dual = build_quadratic_dual(quadratic_A, quadratic_b)
result = solve_and_load(quadratic_dual, nlp_solver)
quadratic_explicit_lambda = np.array([pyo.value(quadratic_dual.lam[i]) for i in quadratic_dual.rows])
quadratic_dual_value = pyo.value(quadratic_dual.objective)
print(f"Termination: {result.solver.termination_condition}")
display(pd.DataFrame(
    {"Explicit Lagrangian dual": quadratic_explicit_lambda,
     "From primal suffix (-d)": quadratic_from_suffix},
    index=pd.Index([f"lambda_{i}" for i in quadratic_dual.rows], name="Multiplier"),
).style.format(lambda value: format_number(value, 6)))
print(f"Dual objective: {quadratic_dual_value:.9f}")
print(f"Primal minus dual objective: {quadratic_primal_value - quadratic_dual_value:.2e}")

Termination: optimal


,Explicit Lagrangian dual,From primal suffix (-d)
Multiplier,,
lambda_1,0.000000,0.000000
lambda_2,0.000000,0.000000
lambda_3,2.000000,2.000000
lambda_4,4.000000,4.000000
lambda_5,0.000040,0.000027


Dual objective: 4.999999999
Primal minus dual objective: 9.44e-10


For checking an expansion by hand, the same dual function is

$$
\begin{aligned}
\ell(\lambda)={}&-\tfrac54\lambda_1^2-\tfrac12\lambda_2^2
-\tfrac14(\lambda_3^2+\lambda_4^2+\lambda_5^2)\\
&-\lambda_1\lambda_2+\lambda_1\lambda_3+\tfrac12\lambda_1\lambda_4
+\tfrac12\lambda_2\lambda_3+\tfrac12\lambda_2\lambda_5\\
&-5\lambda_1-2\lambda_2+\lambda_3+2\lambda_4.
\end{aligned}
$$

This expanded polynomial and the squared-norm expression define the same dual.

### Solution to 2(c)

At $\hat x=(1.25,2.25,0.5)$, every $g_i(\hat x)<0$. The problem is convex and
$\hat x$ is strictly feasible, so Slater's condition gives strong duality.
Since the objective is finite on all of $\mathbb{R}^3$ and the constraints
are affine, a feasible point suffices (refined Slater).

An exact primal–dual certificate is

$$
x^*=(1,2,0),\qquad \lambda^*=(0,0,2,4,0).
$$

The primal value is at least $1^2+2^2=5$ since feasibility requires
$x_1\ge1$, $x_2\ge2$ and $x_3\ge0$. The point $x^*$ is feasible and attains 5.
The stated $\lambda^*$ is dual feasible and also attains 5.
The pair $(x^*,\lambda^*)$ satisfies stationarity $2x^*+A^\top\lambda^*=0$
and complementary slackness $\lambda_i^*g_i(x^*)=0$.

Rows 3, 4 and 5 are binding at $x^*$, but $\lambda_5^*=0$. **Binding does not imply
a strictly positive multiplier.** Rows 1 and 2 are slack and have zero multipliers.

In [10]:
certificate_x = np.array([1.0, 2.0, 0.0])
certificate_lambda = np.array([0.0, 0.0, 2.0, 4.0, 0.0])
certificate_g = quadratic_A @ certificate_x - quadratic_b
display(pd.DataFrame(
    {"Exact g_i(x*)": certificate_g, "Exact lambda_i*": certificate_lambda,
     "Complementary product": certificate_g * certificate_lambda},
    index=pd.Index([f"g_{i}" for i in range(1, 6)], name="Row"),
).style.format(format_number))
print("Exact stationarity residual:", 2 * certificate_x + quadratic_A.T @ certificate_lambda)

numerical_kkt = pd.Series({
    "Primal violation (positive part)": max(0.0, quadratic_g.max()),
    "Dual violation (negative part)": max(0.0, -quadratic_from_suffix.min()),
    "Stationarity infinity norm": np.linalg.norm(
        2 * quadratic_point + quadratic_A.T @ quadratic_from_suffix, ord=np.inf
    ),
    "Complementarity infinity norm": np.linalg.norm(quadratic_from_suffix * quadratic_g, ord=np.inf),
    "Primal-dual objective gap": quadratic_primal_value - quadratic_dual_value,
}, name="Numerical residual")
display(numerical_kkt.to_frame().style.format("{:.2e}"))

,Exact g_i(x*),Exact lambda_i*,Complementary product
Row,,,
g_1,-1.0000,0.0000,0.0000
g_2,-1.0000,0.0000,0.0000
g_3,0.0000,2.0000,0.0000
g_4,0.0000,4.0000,0.0000
g_5,0.0000,0.0000,0.0000


Exact stationarity residual: [0. 0. 0.]


,Numerical residual
Primal violation (positive part),0.00e+00
Dual violation (negative part),0.00e+00
Stationarity infinity norm,3.39e-21
Complementarity infinity norm,3.58e-10
Primal-dual objective gap,9.44e-10


## Exercise 3: fixed decisions, prices and feasibility limits

A production unit has capacity 6 and an opening charge of 10. Its output $f$
costs 1 per unit; backup output $g$ costs 4 per unit. For demand $d\ge0$, consider

$$
\begin{aligned}
\min\quad &10z+f+4g\\
\text{s.t.}\quad &f+g=d,\\
&f\le6z,\qquad f,g\ge0,\qquad z\in\{0,1\}.
\end{aligned}
$$

The supplied builder returns an unsolved model. Use `fixed_design=None` for the
MILP, or 0 or 1 to fix only $z$; $f$ and $g$ remain variables. Setting
`backup=False` fixes $g=0$. Build a fresh model for each experiment.

**(a)** Solve the MILP at $d=3,4,6,7$. Fix $z$ to the design chosen at $d=3$,
apply `relax_integrality(model)`, and attach a dual suffix before solving the
resulting LP at $d=3$ and $d=4$. Compare the demand price with the finite cost
change under this fixed design and with the change when redesign is allowed.
Why are these different comparisons? Checks: the MILP values at 3 and 4 are
12 and 14; the fixed design costs 16 at 4. The MILP value at 7 is 20.

**(b)** Fix $z=1$, as chosen at $d=6$ and $d=7$. Derive its value function
$V_1(d)$ by considering whether $d\le6$. At $d=6$, compute the left and right
differences using $h=10^{-3}$, and compare them with the imported demand dual
$\pi$. Repeat the dual calculation at $d=7$. The capacity rent $\eta$ is the
negative of the raw upper-row dual `model.dual[model.capacity]`. Determine the possible pairs
$(\pi,\eta)$ at the kink. Must a returned dual equal one of the two slopes?
Must binding capacity have positive rent?

**(c)** At $d=5$ with $z=1$ fixed, compare the LP with an equivalent model whose
balance is $-f-g=-d$. Explain the change in the raw equality dual and why the
sensitivity to physical demand is unchanged.

**(d)** Remove backup and keep $z=1$. Solve at $d=6$ and $d=6.1$ using `solve`.
Report both statuses. Compute a right difference only if both solves are optimal;
otherwise report `None`. Interpret the value beyond the feasibility limit.
Never read variable values or duals from an infeasible solve.

**(e), optional.** Let $C\ge0$ be available capacity and exclude the opening
charge. Derive the operating cost $W(C;d)$, then compare the saving from opening
capacity 6 with the charge 10. Find the demand where the two designs tie.
At $C=0,d=3$, derive all optimal demand and capacity multipliers. Compare the
capacity rents with the right derivative of $W$ at zero.

`solve(model)` returns a dictionary containing `status` and `objective`; the
latter is `None` if no feasible solution was loaded. Fixed binary domains still
prevent dual loading: fix the design, relax integrality, then attach the suffix
and solve. Relaxing an unfixed design would give a different problem.

In [11]:
solve = lab_setup.solve
relax_integrality = lab_setup.relax_integrality


def build_dispatch(d, fixed_design=None, backup=True):
    model = pyo.ConcreteModel(name="Production and backup")
    model.z = pyo.Var(domain=pyo.Binary)
    model.f = pyo.Var(domain=pyo.NonNegativeReals)
    model.g = pyo.Var(domain=pyo.NonNegativeReals)
    model.balance = pyo.Constraint(expr=model.f + model.g == d)
    model.capacity = pyo.Constraint(expr=model.f <= 6 * model.z)
    model.objective = pyo.Objective(expr=10 * model.z + model.f + 4 * model.g)
    if fixed_design is not None:
        model.z.fix(fixed_design)
    if not backup:
        model.g.fix(0)
    return model

### Solution to 3(a)

With $z=0$, only backup is available and $V_0(d)=4d$. With $z=1$, use the cheaper
unit up to capacity before using backup. The MILP chooses the less expensive
design. At $d=3$ it chooses $z=0$; fixing this design leaves a demand price of 4.
An increase to $d=4$ therefore costs 4 with the design fixed, whereas redesign
reduces the total increase to 2. The latter is a finite change across a design
switch, rather than an LP derivative of the MILP value function.

In [12]:
dispatch_milps = {}
dispatch_design_rows = []
for demand in [3, 4, 6, 7]:
    model = build_dispatch(demand)
    report = solve(model)
    if report["status"] != "optimal":
        raise RuntimeError(report)
    dispatch_milps[demand] = model
    dispatch_design_rows.append({
        "Demand": demand, "z": pyo.value(model.z),
        "f": pyo.value(model.f), "g": pyo.value(model.g),
        "Cost": report["objective"],
    })
display(pd.DataFrame(dispatch_design_rows).set_index("Demand").style.format(format_number))

chosen_design = int(pyo.value(dispatch_milps[3].z) > 0.5)
dispatch_fixed = {}
dispatch_fixed_rows = []
for demand in [3, 4]:
    model = build_dispatch(demand, fixed_design=chosen_design)
    relax_integrality(model)
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    solve_and_load(model, lp_solver)
    dispatch_fixed[demand] = model
    dispatch_fixed_rows.append({
        "Demand": demand, "Fixed z": chosen_design,
        "Cost": pyo.value(model.objective), "Demand dual": model.dual[model.balance],
    })
display(pd.DataFrame(dispatch_fixed_rows).set_index("Demand").style.format(format_number))

,z,f,g,Cost
Demand,,,,
3,0.0000,0.0000,3.0000,12.0000
4,1.0000,4.0000,0.0000,14.0000
6,1.0000,6.0000,0.0000,16.0000
7,1.0000,6.0000,1.0000,20.0000


,Fixed z,Cost,Demand dual
Demand,,,
3,0.0000,12.0000,4.0000
4,0.0000,16.0000,4.0000


### Solution to 3(b)

For the open design,

$$
V_1(d)=10+d+3\max(d-6,0),\qquad d\ge0.
$$

At $d=6$ the one-sided slopes are 1 and 4. To see the possible multipliers,
use $L=10+f+4g+\pi(d-f-g)+\eta(f-6)$, with $f,g\ge0$ and $\eta\ge0$.
Dual feasibility requires $\pi-\eta\le1$ and $\pi\le4$. Since $f=6>0$,
complementary slackness gives $\eta=\pi-1$. Thus $1\le\pi\le4$ and
$0\le\eta\le3$. Every such pair attains $10+6\pi-6\eta=16$.

The demand dual is a subgradient at the kink. A value strictly between the
one-sided slopes matches neither difference and is still valid. Binding capacity
can have zero rent here. At $d=7$, both outputs are positive, forcing
$\pi=4$ and $\eta=3$; the raw upper-row dual is $-3$.

In [13]:
dispatch_open = {}
dispatch_price_rows = []
for demand in [6, 7]:
    design = int(pyo.value(dispatch_milps[demand].z) > 0.5)
    model = build_dispatch(demand, fixed_design=design)
    relax_integrality(model)
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    solve_and_load(model, lp_solver)
    dispatch_open[demand] = model
    dispatch_price_rows.append({
        "Demand": demand, "Cost": pyo.value(model.objective),
        "pi": model.dual[model.balance], "Raw capacity dual": model.dual[model.capacity],
        "eta": -model.dual[model.capacity],
    })
display(pd.DataFrame(dispatch_price_rows).set_index("Demand").style.format(format_number))

dispatch_h = 1e-3
dispatch_near_kink = {}
for demand in [6 - dispatch_h, 6, 6 + dispatch_h]:
    model = build_dispatch(demand, fixed_design=1)
    relax_integrality(model)
    solve_and_load(model, lp_solver)
    dispatch_near_kink[demand] = pyo.value(model.objective)
dispatch_left = (dispatch_near_kink[6] - dispatch_near_kink[6 - dispatch_h]) / dispatch_h
dispatch_right = (dispatch_near_kink[6 + dispatch_h] - dispatch_near_kink[6]) / dispatch_h
print(f"Left slope: {dispatch_left:.6f}; right slope: {dispatch_right:.6f}")

,Cost,pi,Raw capacity dual,eta
Demand,,,,
6,16.0000,1.0000,0.0000,0.0000
7,20.0000,4.0000,-3.0000,3.0000


Left slope: 1.000000; right slope: 4.000000


### Solution to 3(c)

At $d=5$, the open design has marginal cost 1. The original row has RHS $d$,
so its raw dual is 1. The negated row has RHS $-d$ and raw dual $-1$.
Sensitivity to physical demand is the raw dual times the derivative of that
row's RHS with respect to $d$, giving $(-1)(-1)=1$ in the second model.

In [14]:
dispatch_sign_rows = []
for flipped in [False, True]:
    model = build_dispatch(5, fixed_design=1)
    if flipped:
        model.balance.set_value(-model.f - model.g == -5)
    relax_integrality(model)
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    solve_and_load(model, lp_solver)
    raw_dual = model.dual[model.balance]
    dispatch_sign_rows.append({
        "Balance": "-f-g=-d" if flipped else "f+g=d",
        "Cost": pyo.value(model.objective), "Raw demand dual": raw_dual,
        "Physical demand price": (-1 if flipped else 1) * raw_dual,
    })
display(pd.DataFrame(dispatch_sign_rows).set_index("Balance").style.format(format_number))

,Cost,Raw demand dual,Physical demand price
Balance,,,
f+g=d,15.0000,1.0000,1.0000
-f-g=-d,15.0000,-1.0000,1.0000


### Solution to 3(d)

Without backup, $f=d\le6$. Demand 6 is feasible and costs 16; demand 6.1 is
infeasible. There is no finite numerical right difference to report. Extending
the value function by $+\infty$ outside its feasible domain gives an infinite
right difference here. A finite multiplier at the boundary cannot certify
feasibility of an increased demand.

In [15]:
dispatch_without_backup = {}
for demand in [6, 6.1]:
    model = build_dispatch(demand, fixed_design=1, backup=False)
    relax_integrality(model)
    dispatch_without_backup[demand] = solve(model)
    report = dispatch_without_backup[demand]
    print(f"Demand {demand}: status={report['status']}, objective={report['objective']}")

dispatch_right_difference = None
if all(report["status"] == "optimal" for report in dispatch_without_backup.values()):
    dispatch_right_difference = (
        dispatch_without_backup[6.1]["objective"] - dispatch_without_backup[6]["objective"]
    ) / 0.1
print("Numerical right difference:", dispatch_right_difference)

Demand 6: status=optimal, objective=16.0


Demand 6.1: status=infeasible, objective=None
Numerical right difference: None


### Solution to 3(e), optional

The cheaper unit supplies $\min(C,d)$, so
$W(C;d)=4d-3\min(C,d)$. Opening capacity 6 saves $3\min(6,d)$ in operating
cost. The designs therefore tie at $3d=10$, or $d=10/3$.

The operating dual is
$\max\{d\pi-C\eta:\pi-\eta\le1,\ \pi\le4,\ \eta\ge0\}$.
At $C=0,d=3$, every pair $\pi=4$, $\eta\ge3$ is optimal. The right derivative
of $W$ with respect to capacity is $-3$, which need not equal an arbitrary
returned $-\eta$. The finite saving above follows from the explicit value
function; extrapolating a single boundary multiplier would not establish it.

## Exercise 4: maximum flow, competing cuts and integrality

Use Network 2 from week 1, supplied again below, with source 0 and sink 10.
We now seek maximum throughput: replace the original supplies and demands by
outgoing minus incoming flow equal to $v$ at the source, $-v$ at the sink and
zero at every intermediate node. Arc capacities are unchanged; costs are unused.

**(a)** Implement `build_max_flow(network, source, sink)` with continuous
nonnegative flows and throughput $v$, maximizing $v$. Keep capacity limits as
explicit rows and attach a dual suffix. Solve with HiGHS and report the flows,
throughput and capacity duals. Check: the maximum throughput is 15.

**(b)** The residual graph has forward capacity $u_{ij}-f_{ij}$ and backward
capacity $f_{ij}$ on each arc. The supplied function searches this graph.
Call it from the source to obtain a source side
$S_1$. Call it from the sink with `reverse=True` to find nodes that can reach the
sink; take their complement to obtain another source side $S_2$. For each side,
list the original arcs leaving it and sum their capacities directly from the
data. Compare these bounds with the throughput and with the nonzero capacity
duals. Explain the connection to strong LP duality. Does one returned dual
identify every minimum cut?

**(c)** Predict the new throughput when increasing the capacity of $(0,1)$ by
1; of both $(0,1)$ and $(1,2)$ by 1; and of both by 10. Solve a fresh model for
each case, always starting from the original data. Find a cut explaining the
last result. A saturated arc alone need not be the only bottleneck.

**(d)** Inspect the flows in (a) and recall the continuous flow models in week 1.
State the integrality theorem for a directed node–arc incidence matrix, including
its assumptions on balances and bounds. Then solve the continuous LP

$$
\max\ x_1+x_2+x_3\quad\text{s.t.}\quad
x_1+x_2\le1,\quad x_2+x_3\le1,\quad x_1+x_3\le1,\quad x\ge0.
$$

Show that its maximum is attained at a unique fractional vertex and find the
best integral value by hand. Minimize the same objective over the same feasible
set. Does an integral optimizer for this second objective establish that the
polytope is integral? Explain why the directed-incidence theorem does not apply
to this triangle matrix, whose columns each contain two $+1$ entries.

In [16]:
flow_network = {
    "nodes": list(range(11)),
    "edges": {
        (0, 1): {"u": 15}, (1, 2): {"u": 15}, (2, 9): {"u": 10},
        (2, 3): {"u": 15}, (3, 4): {"u": 10}, (3, 5): {"u": 10},
        (4, 7): {"u": 4}, (4, 8): {"u": 5}, (5, 8): {"u": 5},
        (5, 9): {"u": 6}, (5, 10): {"u": 5}, (8, 6): {"u": 8},
        (6, 10): {"u": 1}, (7, 8): {"u": 4}, (7, 10): {"u": 2},
        (8, 10): {"u": 5}, (9, 10): {"u": 3},
    },
}
flow_source, flow_sink = 0, 10


def residual_reachable(network, flows, start, reverse=False, tol=1e-7):
    '''Find residual-reachable nodes; reverse=True finds nodes that can reach start.

    tol is an absolute tolerance in flow units. Supply a feasible flow.
    '''
    adjacency = {node: set() for node in network["nodes"]}
    for (i, j), data in network["edges"].items():
        if data["u"] - flows[i, j] > tol:
            adjacency[i].add(j)
        if flows[i, j] > tol:
            adjacency[j].add(i)
    if reverse:
        reversed_adjacency = {node: set() for node in network["nodes"]}
        for i, neighbors in adjacency.items():
            for j in neighbors:
                reversed_adjacency[j].add(i)
        adjacency = reversed_adjacency
    reached = {start}
    pending = [start]
    while pending:
        node = pending.pop()
        for neighbor in adjacency[node] - reached:
            reached.add(neighbor)
            pending.append(neighbor)
    return reached

### Solution to 4(a)

Every intermediate balance is zero. Throughput is a decision, rather than the
old supply of 15. The capacity duals below belong to a maximization LP: their
sign is nonnegative, since increasing an upper capacity cannot reduce the
maximum throughput. The earlier minimization sign table must be adjusted for
this objective sense.

In [17]:
def build_max_flow(network, source, sink):
    model = pyo.ConcreteModel(name="Maximum flow")
    model.nodes = pyo.Set(initialize=network["nodes"])
    model.arcs = pyo.Set(initialize=list(network["edges"]), dimen=2)
    model.flow = pyo.Var(model.arcs, domain=pyo.NonNegativeReals)
    model.v = pyo.Var(domain=pyo.NonNegativeReals)
    model.objective = pyo.Objective(expr=model.v, sense=pyo.maximize)

    @model.Constraint(model.nodes)
    def balance(m, node):
        outgoing = sum(m.flow[i, j] for i, j in m.arcs if i == node)
        incoming = sum(m.flow[i, j] for i, j in m.arcs if j == node)
        rhs = m.v if node == source else -m.v if node == sink else 0
        return outgoing - incoming == rhs

    @model.Constraint(model.arcs)
    def capacity(m, i, j):
        return m.flow[i, j] <= network["edges"][i, j]["u"]

    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return model

max_flow_model = build_max_flow(flow_network, flow_source, flow_sink)
solve_and_load(max_flow_model, lp_solver)
max_flow_value = pyo.value(max_flow_model.v)
max_flow_values = {edge: pyo.value(max_flow_model.flow[edge]) for edge in flow_network["edges"]}
display(pd.DataFrame([
    {"Arc": str(edge), "Capacity": data["u"], "Flow": max_flow_values[edge],
     "Capacity dual": max_flow_model.dual[max_flow_model.capacity[edge]]}
    for edge, data in flow_network["edges"].items()
]).set_index("Arc").style.format(format_number))
print(f"Maximum throughput: {max_flow_value:.6f}")

,Capacity,Flow,Capacity dual
Arc,,,
"(0, 1)",15.0000,15.0000,1.0000
"(1, 2)",15.0000,15.0000,0.0000
"(2, 9)",10.0000,3.0000,0.0000
"(2, 3)",15.0000,12.0000,0.0000
"(3, 4)",10.0000,7.0000,0.0000
"(3, 5)",10.0000,5.0000,0.0000
"(4, 7)",4.0000,2.0000,0.0000
"(4, 8)",5.0000,5.0000,0.0000
"(5, 8)",5.0000,0.0000,0.0000


Maximum throughput: 15.000000


### Solution to 4(b)

For any source side $S$ containing 0 but not 10, summing its balances gives

$$
v=\sum_{i\in S,\,j\notin S}f_{ij}
-\sum_{i\notin S,\,j\in S}f_{ij}
\le\sum_{i\in S,\,j\notin S}u_{ij}=U(S).
$$

The LP dual minimizes $\sum_{ij}u_{ij}q_{ij}$ subject to
$q_{ij}\ge p_i-p_j$, $q_{ij}\ge0$ and $p_0-p_{10}\ge1$, with free node
potentials $p$. Choosing $p_i=1$ on $S$ and 0 elsewhere, with $q=1$ on arcs
leaving $S$ and 0 elsewhere, makes every cut a dual-feasible upper bound.
The residual cuts below attain the primal throughput, providing matching
primal–dual certificates. This is max-flow/min-cut in this instance.

The two source sides are $\{0\}$ and $\{0,1\}$, each with capacity 15. A dual
solution can place its capacity multiplier on either bottleneck; one returned
dual vector need not expose all minimum cuts.

In [18]:
flow_source_sides = [
    residual_reachable(flow_network, max_flow_values, flow_source),
    set(flow_network["nodes"]) - residual_reachable(
        flow_network, max_flow_values, flow_sink, reverse=True
    ),
]
flow_cut_rows = []
for side in flow_source_sides:
    leaving = [(i, j) for i, j in flow_network["edges"] if i in side and j not in side]
    capacity = sum(flow_network["edges"][edge]["u"] for edge in leaving)
    flow_cut_rows.append({
        "Source side": str(sorted(side)), "Leaving arcs": str(leaving),
        "Cut capacity": capacity, "Capacity minus throughput": capacity - max_flow_value,
    })
display(pd.DataFrame(flow_cut_rows).set_index("Source side"))

,Leaving arcs,Cut capacity,Capacity minus throughput
Source side,,,
[0],"[(0, 1)]",15,0.0
"[0, 1]","[(1, 2)]",15,0.0


### Solution to 4(c)

Raising only $u_{01}$ leaves the cut $S=\{0,1\}$ at 15. Raising both initial
capacities by 1 permits 16. Raising both by 10 still permits only 16: the cut
with every node except 10 on its source side has capacity
$u_{5,10}+u_{6,10}+u_{7,10}+u_{8,10}+u_{9,10}=5+1+2+5+3=16$.
The solved flows attain these upper bounds. Extrapolating one returned
capacity dual over a finite change would miss a competing cut.

In [19]:
flow_perturbations = [
    ("Original", {}),
    ("First arc +1", {(0, 1): 1}),
    ("Both arcs +1", {(0, 1): 1, (1, 2): 1}),
    ("Both arcs +10", {(0, 1): 10, (1, 2): 10}),
]
flow_perturbed_models = {}
flow_perturbation_rows = []
for label, changes in flow_perturbations:
    network = {
        "nodes": list(flow_network["nodes"]),
        "edges": {edge: dict(data) for edge, data in flow_network["edges"].items()},
    }
    for edge, increase in changes.items():
        network["edges"][edge]["u"] += increase
    model = build_max_flow(network, flow_source, flow_sink)
    solve_and_load(model, lp_solver)
    flow_perturbed_models[label] = model
    flow_perturbation_rows.append({"Case": label, "Throughput": pyo.value(model.v)})
display(pd.DataFrame(flow_perturbation_rows).set_index("Case").style.format(format_number))

,Throughput
Case,
Original,15.0000
First arc +1,15.0000
Both arcs +1,16.0000
Both arcs +10,16.0000


### Solution to 4(d)

A directed node–arc incidence matrix has one $+1$ and one $-1$ per column and
is totally unimodular. With integral balances and integral finite bounds, every
vertex of $\{f:Af=b,\ 0\le f\le u\}$ is integral. A finite optimum is attained
at a vertex, so continuous flows suffice to obtain an integral optimal solution.
The maximum-flow model has the same structure if $v$ is viewed as flow on an
added return arc from 10 to 0. Observing integer outputs alone would not prove
this theorem.

For the triangle, summing the three rows gives $2(x_1+x_2+x_3)\le3$.
Equality forces every row to bind, giving the unique maximizer
$(1/2,1/2,1/2)$ and value $3/2$. At most one coordinate of an integral feasible
point can equal 1, so its best value is 1. Minimizing gives the unique optimizer
0. The fractional vertex remains in the feasible set under either objective.

The triangle's constraint matrix has determinant 2, so it is not totally
unimodular. Integral RHS values and one integral optimizer do not establish
integrality of a polytope.

In [20]:
triangle_models = {}
triangle_rows = []
for label, sense in [("Maximum", pyo.maximize), ("Minimum", pyo.minimize)]:
    model = pyo.ConcreteModel(name="Triangle allocation")
    model.x = pyo.Var([1, 2, 3], domain=pyo.NonNegativeReals)
    model.pairs = pyo.Set(initialize=[(1, 2), (2, 3), (1, 3)], dimen=2)

    @model.Constraint(model.pairs)
    def pair_limit(m, i, j):
        return m.x[i] + m.x[j] <= 1

    model.objective = pyo.Objective(expr=sum(model.x[i] for i in [1, 2, 3]), sense=sense)
    solve_and_load(model, lp_solver)
    triangle_models[label] = model
    triangle_rows.append({
        "Sense": label, **{f"x{i}": pyo.value(model.x[i]) for i in [1, 2, 3]},
        "Value": pyo.value(model.objective),
    })
display(pd.DataFrame(triangle_rows).set_index("Sense").style.format(format_number))

,x1,x2,x3,Value
Sense,,,,
Maximum,0.5000,0.5000,0.5000,1.5000
Minimum,0.0000,0.0000,0.0000,0.0000


## Exercise 5: the dual of a bounded allocation LP (optional)

Three divisible activities have rewards $c=(4,3,1)$ and resource requirements
$w=(1,2,1)$. Choose fractions $\xi_i$ subject to a resource budget $B\ge0$:

$$
\max\ c^\top\xi\quad\text{s.t.}\quad
w^\top\xi\le B,\qquad 0\le\xi_i\le1\quad(i=1,2,3).
$$

**(a)** Derive the dual before coding it. Use a multiplier $\lambda$ for the
budget and $\mu_i$ for each upper bound. Explain their signs, the dual
constraints and objective. Justify strong duality by feasibility and boundedness.

**(b)** Implement both LPs with builders that take $c,w,B$ as arguments. At
$B=2$, compare primal and dual feasibility, objective values and complementary
slackness. Keep the allocation variables continuous. Check: the value is 5.5.
Interpret $\lambda$ as the marginal value of additional resource near this budget.

**(c)** Repeat at $B=0$ and $B=\sum_iw_i=4$; the values are 0 and 8. Describe
all optimal dual multipliers at each endpoint. Compare feasibility and objective
values, allowing different solvers to select different optimal multipliers.

In [21]:
allocation_c = {1: 4, 2: 3, 3: 1}
allocation_w = {1: 1, 2: 2, 3: 1}

### Solution to 5(a)

For $\lambda,\mu\ge0$, the maximization Lagrangian is

$$
L(\xi,\lambda,\mu)=c^\top\xi+\lambda(B-w^\top\xi)
+\sum_i\mu_i(1-\xi_i)
=B\lambda+\sum_i\mu_i+\sum_i(c_i-w_i\lambda-\mu_i)\xi_i.
$$

Its supremum over $\xi\ge0$ is finite exactly when
$w_i\lambda+\mu_i\ge c_i$ for every $i$. The dual is therefore

$$
\min\ B\lambda+\sum_i\mu_i\quad\text{s.t.}\quad
w_i\lambda+\mu_i\ge c_i,\qquad\lambda,\mu\ge0.
$$

The primal is feasible at $\xi=0$ for every $B\ge0$ and bounded by its box
constraints. The dual is feasible, for example at $\lambda=0$, $\mu=c$.
LP strong duality applies, including at the two budget endpoints.

In [22]:
def build_allocation_primal(c, w, B):
    model = pyo.ConcreteModel(name="Bounded allocation primal")
    model.activities = pyo.Set(initialize=list(c))
    model.xi = pyo.Var(model.activities, bounds=(0, 1))
    model.budget = pyo.Constraint(expr=sum(w[i] * model.xi[i] for i in model.activities) <= B)
    model.objective = pyo.Objective(
        expr=sum(c[i] * model.xi[i] for i in model.activities), sense=pyo.maximize
    )
    return model


def build_allocation_dual(c, w, B):
    model = pyo.ConcreteModel(name="Bounded allocation dual")
    model.activities = pyo.Set(initialize=list(c))
    model.lam = pyo.Var(domain=pyo.NonNegativeReals)
    model.mu = pyo.Var(model.activities, domain=pyo.NonNegativeReals)

    @model.Constraint(model.activities)
    def cover(m, i):
        return w[i] * m.lam + m.mu[i] >= c[i]

    model.objective = pyo.Objective(expr=B * model.lam + sum(model.mu[i] for i in model.activities))
    return model

### Solution to 5(b)–(c)

At $B=2$, a certificate is $\xi=(1,1/2,0)$, $\lambda=3/2$ and
$\mu=(5/2,0,0)$; both values are $11/2$. Complementary slackness requires

$$
\lambda(B-w^\top\xi)=0,\qquad
\mu_i(1-\xi_i)=0,\qquad
\xi_i(w_i\lambda+\mu_i-c_i)=0.
$$

The partially used second activity gives $w_2\lambda=c_2$, so
$\lambda=c_2/w_2=3/2$. A small increase in budget is worth $3/2$ per unit.

At $B=0$, positivity of $w$ forces $\xi=0$. Every optimal dual has $\mu=0$
and $\lambda\ge\max_i(c_i/w_i)=4$. At $B=4$, all activities can be fully used.
Every pair $0\le\lambda\le\min_i(c_i/w_i)=1$,
$\mu_i=c_i-w_i\lambda$ is dual optimal and attains 8. These families describe
all endpoint dual optima; a single returned vector is only one member.

In [23]:
allocation_models = {}
allocation_rows = []
allocation_check_rows = []
for budget in [0, 2, 4]:
    primal = build_allocation_primal(allocation_c, allocation_w, budget)
    dual = build_allocation_dual(allocation_c, allocation_w, budget)
    solve_and_load(primal, lp_solver)
    solve_and_load(dual, lp_solver)
    allocation_models[budget] = (primal, dual)
    xi = np.array([pyo.value(primal.xi[i]) for i in primal.activities])
    lam = pyo.value(dual.lam)
    mu = np.array([pyo.value(dual.mu[i]) for i in primal.activities])
    c = np.array([allocation_c[i] for i in primal.activities])
    w = np.array([allocation_w[i] for i in primal.activities])
    budget_slack = budget - w @ xi
    cover_slack = w * lam + mu - c
    allocation_rows.append({
        "B": budget, "Primal value": pyo.value(primal.objective),
        "Dual value": pyo.value(dual.objective), "xi": tuple(xi),
        "lambda": lam, "mu": tuple(mu),
    })
    allocation_check_rows.append({
        "B": budget,
        "Primal violation": max(0.0, -xi.min(), (xi - 1).max(), -budget_slack),
        "Dual violation": max(0.0, -lam, -mu.min(), -cover_slack.min()),
        "Objective gap": pyo.value(dual.objective) - pyo.value(primal.objective),
        "Complementarity residual": max(
            abs(lam * budget_slack), np.max(np.abs(mu * (1 - xi))),
            np.max(np.abs(xi * cover_slack)),
        ),
    })
display(pd.DataFrame(allocation_rows).set_index("B").style.format({
    "Primal value": format_number, "Dual value": format_number, "lambda": format_number,
    "xi": lambda values: "(" + ", ".join(format_number(value) for value in values) + ")",
    "mu": lambda values: "(" + ", ".join(format_number(value) for value in values) + ")",
}))
display(pd.DataFrame(allocation_check_rows).set_index("B").style.format("{:.2e}"))

,Primal value,Dual value,xi,lambda,mu
B,,,,,
0,0.0000,0.0000,"(0.0000, 0.0000, 0.0000)",4.0000,"(0.0000, 0.0000, 0.0000)"
2,5.5000,5.5000,"(1.0000, 0.5000, 0.0000)",1.5000,"(2.5000, 0.0000, 0.0000)"
4,8.0000,8.0000,"(1.0000, 1.0000, 1.0000)",0.0000,"(4.0000, 3.0000, 1.0000)"


,Primal violation,Dual violation,Objective gap,Complementarity residual
B,,,,
0,0.00e+00,0.00e+00,0.00e+00,0.00e+00
2,0.00e+00,0.00e+00,0.00e+00,0.00e+00
4,0.00e+00,0.00e+00,0.00e+00,0.00e+00


What did your checks establish, which errors could they detect, and what remains
unchecked?